# CleanAir AI — Notebook 04: AQI Forecasting Models

This notebook trains and compares multiple AQI forecasting models using real AQI history collected from the CPCB/data.gov.in API.

## Inputs

- `data/processed/station_aqi_history.csv`
- `data/processed/station_source_attributed.csv`

## Main tasks

1. Load real AQI history.
2. Validate whether enough time-series data exists.
3. Create target variable for next-timestamp AQI prediction.
4. Create lag, rolling, weather, source, and contextual features.
5. Train multiple forecasting models.
6. Compare models using a time-based 80/20 split.
7. Select the best model using lowest MAE.
8. Evaluate category prediction and high-AQI detection.
9. Save trained models, predictions, feature importance, and final metrics.

## Models tested

- Persistence Baseline
- Linear Regression
- Ridge Regression
- Random Forest
- Extra Trees
- Gradient Boosting
- HistGradientBoosting
- XGBoost, if available

In [1]:
from pathlib import Path
from datetime import datetime
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

from sklearn.inspection import permutation_importance

import joblib

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [
    PROCESSED_DIR,
    MODELS_DIR,
    OUTPUTS_DIR,
    REPORTS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed folder:", PROCESSED_DIR)
print("Models folder:", MODELS_DIR)
print("Outputs folder:", OUTPUTS_DIR)
print("Reports folder:", REPORTS_DIR)

Project root: C:\Users\Lenovo\Desktop\CleanAir_AI
Processed folder: C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed
Models folder: C:\Users\Lenovo\Desktop\CleanAir_AI\models
Outputs folder: C:\Users\Lenovo\Desktop\CleanAir_AI\outputs
Reports folder: C:\Users\Lenovo\Desktop\CleanAir_AI\reports


In [3]:
HISTORY_PATH = PROCESSED_DIR / "station_aqi_history.csv"

if not HISTORY_PATH.exists():
    raise FileNotFoundError(
        f"Missing file: {HISTORY_PATH}. Run Notebook 01 first."
    )

history_df = pd.read_csv(HISTORY_PATH)

print("Loaded AQI history.")
print("Shape:", history_df.shape)

display(history_df.head())

Loaded AQI history.
Shape: (6072, 25)


,station_id,station_name,city,state,lat,lon,timestamp,snapshot_fetch_time,PM2.5,PM10,...,aqi,aqi_category,dominant_pollutant,valid_pollutant_count,has_particulate,aqi_is_valid,aqi_quality_flag,reported_aqi,reported_aqi_category,reported_dominant_pollutant
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 18:00:00,2026-07-15 19:50:36,46.0,135.0,...,425.28,Severe,CO,7.0,1.0,1.0,Valid AQI,425.28,Severe,CO
1,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 19:00:00,2026-07-15 20:21:44,46.0,135.0,...,443.96,Severe,CO,7.0,1.0,1.0,Valid AQI,443.96,Severe,CO
2,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 20:00:00,2026-07-15 21:23:14,47.0,136.0,...,500.00,Severe,CO,7.0,1.0,1.0,Valid AQI,500.00,Severe,CO
3,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 21:00:00,2026-07-15 22:25:46,47.0,139.0,...,126.25,Moderate,PM10,3.0,1.0,1.0,Valid AQI,126.25,Moderate,PM10
4,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 22:00:00,2026-07-15 23:43:01,48.0,144.0,...,500.00,Severe,CO,7.0,1.0,1.0,Valid AQI,500.00,Severe,CO


In [4]:
SOURCE_ATTRIBUTED_PATH = PROCESSED_DIR / "station_source_attributed.csv"

if not SOURCE_ATTRIBUTED_PATH.exists():
    raise FileNotFoundError(
        f"Missing file: {SOURCE_ATTRIBUTED_PATH}. Run Notebook 03 first."
    )

source_df = pd.read_csv(SOURCE_ATTRIBUTED_PATH)

print("Loaded source-attributed station data.")
print("Shape:", source_df.shape)

display(source_df.head())

Loaded source-attributed station data.
Shape: (500, 50)


,station_id,station_name,city,state,lat,lon,timestamp,PM2.5,PM10,NO2,...,primary_pollution_source,primary_source_score,secondary_source_score,source_score_gap,source_attribution_confidence,source_attribution_confidence_category,source_evidence,landuse_source_alignment,recommended_intervention,notebook_03_processed_time
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-16 05:00:00,54.0,172.0,32.0,...,Construction / road dust,38.40,34.56,3.84,28.03,Low confidence,PM10-heavy particulate pattern; road dust / co...,Partially aligned,High priority: Prioritize road dust suppressio...,2026-07-16 10:55:13
1,andhra_pradesh__anantapur__gulzarpet_anantapur...,"Gulzarpet, Anantapur - APPCB",Anantapur,Andhra Pradesh,14.675886,77.593027,2026-07-16 05:00:00,85.0,76.0,16.0,...,Biomass / fine-particle combustion,35.37,29.84,5.53,26.42,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
2,andhra_pradesh__chittoor__gangineni_cheruvu_ch...,"Gangineni Cheruvu, Chittoor - APPCB",Chittoor,Andhra Pradesh,13.204880,79.097889,2026-07-16 05:00:00,70.0,64.0,31.0,...,Biomass / fine-particle combustion,36.34,33.72,2.62,26.22,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
3,andhra_pradesh__eluru__district_court_eluru_appcb,"District Court, Eluru - APPCB",Eluru,Andhra Pradesh,16.711754,81.092095,2026-07-16 05:00:00,49.0,49.0,22.0,...,Biomass / fine-particle combustion,34.09,26.13,7.96,26.25,Low confidence,PM2.5-heavy fine-particle pattern; CO and stag...,Partially aligned,High priority: Prioritize open-burning control...,2026-07-16 10:55:13
4,andhra_pradesh__guntur__rajendra_nagar_north_g...,"Rajendra Nagar North, Guntur - APPCB",Guntur,Andhra Pradesh,16.316553,80.413302,2026-07-16 05:00:00,52.0,52.0,42.0,...,Photochemical smog,32.25,31.11,1.14,22.92,Low confidence,O3 and temperature-related signal; urban photo...,Partially aligned,High priority: Prioritize NOx/VOC precursor co...,2026-07-16 10:55:13


In [5]:
required_history_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "lat",
    "lon",
    "timestamp",
    "reported_aqi",
    "reported_aqi_category",
    "reported_dominant_pollutant"
]

missing_columns = [
    col for col in required_history_columns
    if col not in history_df.columns
]

if missing_columns:
    raise KeyError(f"Missing required history columns: {missing_columns}")

history_df["timestamp"] = pd.to_datetime(
    history_df["timestamp"],
    errors="coerce"
)

history_df["reported_aqi"] = pd.to_numeric(
    history_df["reported_aqi"],
    errors="coerce"
)

history_df = history_df.dropna(
    subset=[
        "station_id",
        "timestamp",
        "reported_aqi"
    ]
).copy()

history_df = history_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("History validation complete.")
print("Clean history shape:", history_df.shape)
print("Unique stations:", history_df["station_id"].nunique())
print("Unique timestamps:", history_df["timestamp"].nunique())
print("Date range:", history_df["timestamp"].min(), "to", history_df["timestamp"].max())

display(
    history_df["timestamp"]
    .value_counts()
    .sort_index()
)

History validation complete.
Clean history shape: (5644, 25)
Unique stations: 488
Unique timestamps: 12
Date range: 2026-07-15 18:00:00 to 2026-07-16 05:00:00


timestamp
2026-07-15 18:00:00    463
2026-07-15 19:00:00    468
2026-07-15 20:00:00    470
2026-07-15 21:00:00    468
2026-07-15 22:00:00    464
2026-07-15 23:00:00    468
2026-07-16 00:00:00    470
2026-07-16 01:00:00    472
2026-07-16 02:00:00    472
2026-07-16 03:00:00    474
2026-07-16 04:00:00    477
2026-07-16 05:00:00    478
Name: count, dtype: int64

In [6]:
MIN_UNIQUE_TIMESTAMPS = 12
MIN_TOTAL_ROWS = 1000

unique_timestamps = history_df["timestamp"].nunique()
total_rows = len(history_df)

enough_history = (
    unique_timestamps >= MIN_UNIQUE_TIMESTAMPS
    and total_rows >= MIN_TOTAL_ROWS
)

print("Unique timestamps:", unique_timestamps)
print("Total rows:", total_rows)
print("Enough history for real forecasting:", enough_history)

if not enough_history:
    insufficient_report = {
        "notebook": "04_ml_forecasting_models.ipynb",
        "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "status": "Not enough real time-series history for forecasting",
        "unique_timestamps_available": int(unique_timestamps),
        "total_rows_available": int(total_rows),
        "minimum_unique_timestamps_required": int(MIN_UNIQUE_TIMESTAMPS),
        "minimum_total_rows_required": int(MIN_TOTAL_ROWS),
        "recommendation": (
            "Run Notebook 01 repeatedly over time so that station_aqi_history.csv "
            "accumulates multiple real timestamps per station."
        )
    }

    INSUFFICIENT_REPORT_PATH = REPORTS_DIR / "notebook_04_insufficient_history_report.json"

    with open(INSUFFICIENT_REPORT_PATH, "w", encoding="utf-8") as file:
        json.dump(insufficient_report, file, indent=4)

    print("Insufficient-history report saved:")
    print(INSUFFICIENT_REPORT_PATH)

    print(json.dumps(insufficient_report, indent=4))

Unique timestamps: 12
Total rows: 5644
Enough history for real forecasting: True


In [7]:
if not enough_history:
    raise RuntimeError(
        "Stopping Notebook 04 safely: not enough real AQI history yet."
    )

print("Sufficient history available. Continuing to multi-model forecasting.")

Sufficient history available. Continuing to multi-model forecasting.


In [8]:
context_columns = [
    "station_id",
    "temperature_2m",
    "relative_humidity_2m",
    "rain",
    "wind_speed_10m",
    "cloud_cover",
    "weather_trapping_score",
    "urban_pressure_score",
    "road_density_proxy",
    "environmental_risk_score",
    "primary_pollution_source",
    "source_attribution_confidence",
    "source_attribution_confidence_category",
    "landuse_type_proxy"
]

available_context_columns = [
    col for col in context_columns
    if col in source_df.columns
]

context_df = source_df[available_context_columns].drop_duplicates(
    subset=["station_id"]
)

model_df = history_df.merge(
    context_df,
    on="station_id",
    how="left"
)

print("Merged AQI history with context features.")
print("Shape:", model_df.shape)

display(model_df.head())

Merged AQI history with context features.
Shape: (5644, 38)


,station_id,station_name,city,state,lat,lon,timestamp,snapshot_fetch_time,PM2.5,PM10,...,wind_speed_10m,cloud_cover,weather_trapping_score,urban_pressure_score,road_density_proxy,environmental_risk_score,primary_pollution_source,source_attribution_confidence,source_attribution_confidence_category,landuse_type_proxy
0,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 18:00:00,2026-07-15 19:50:36,46.0,135.0,...,17.7,100.0,55.79,15.0,20.0,51.86,Construction / road dust,28.03,Low confidence,General urban / semi-urban proxy
1,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 19:00:00,2026-07-15 20:21:44,46.0,135.0,...,17.7,100.0,55.79,15.0,20.0,51.86,Construction / road dust,28.03,Low confidence,General urban / semi-urban proxy
2,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 20:00:00,2026-07-15 21:23:14,47.0,136.0,...,17.7,100.0,55.79,15.0,20.0,51.86,Construction / road dust,28.03,Low confidence,General urban / semi-urban proxy
3,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 21:00:00,2026-07-15 22:25:46,47.0,139.0,...,17.7,100.0,55.79,15.0,20.0,51.86,Construction / road dust,28.03,Low confidence,General urban / semi-urban proxy
4,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,16.515083,80.518167,2026-07-15 22:00:00,2026-07-15 23:43:01,48.0,144.0,...,17.7,100.0,55.79,15.0,20.0,51.86,Construction / road dust,28.03,Low confidence,General urban / semi-urban proxy


In [9]:
model_df = model_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

model_df["target_aqi_next"] = (
    model_df
    .groupby("station_id")["reported_aqi"]
    .shift(-1)
)

print("Rows before dropping target NaN:", len(model_df))

model_df = model_df.dropna(
    subset=["target_aqi_next"]
).copy()

print("Rows after dropping target NaN:", len(model_df))

display(
    model_df[
        [
            "station_id",
            "timestamp",
            "reported_aqi",
            "target_aqi_next"
        ]
    ].head(20)
)

Rows before dropping target NaN: 5644
Rows after dropping target NaN: 5156


,station_id,timestamp,reported_aqi,target_aqi_next
0,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 18:00:00,425.28,443.96
1,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 19:00:00,443.96,500.00
2,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 20:00:00,500.00,126.25
3,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 21:00:00,126.25,500.00
4,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 22:00:00,500.00,500.00
5,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-15 23:00:00,500.00,500.00
6,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-16 00:00:00,500.00,500.00
7,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-16 01:00:00,500.00,500.00
8,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-16 02:00:00,500.00,500.00
9,andhra_pradesh__amaravati__secretariat_amarava...,2026-07-16 03:00:00,500.00,500.00


In [10]:
lag_steps = [1, 2, 3, 6, 12, 24]

for lag in lag_steps:
    model_df[f"aqi_lag_{lag}"] = (
        model_df
        .groupby("station_id")["reported_aqi"]
        .shift(lag)
    )

print("Lag features created:")
print([f"aqi_lag_{lag}" for lag in lag_steps])

Lag features created:
['aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_6', 'aqi_lag_12', 'aqi_lag_24']


In [11]:
rolling_windows = [3, 6, 12, 24]

for window in rolling_windows:
    model_df[f"aqi_rolling_mean_{window}"] = (
        model_df
        .groupby("station_id")["reported_aqi"]
        .transform(
            lambda s: s.shift(1).rolling(
                window=window,
                min_periods=1
            ).mean()
        )
    )

    model_df[f"aqi_rolling_std_{window}"] = (
        model_df
        .groupby("station_id")["reported_aqi"]
        .transform(
            lambda s: s.shift(1).rolling(
                window=window,
                min_periods=2
            ).std()
        )
    )

print("Rolling features created.")

Rolling features created.


In [12]:
model_df["hour"] = model_df["timestamp"].dt.hour
model_df["dayofweek"] = model_df["timestamp"].dt.dayofweek
model_df["month"] = model_df["timestamp"].dt.month
model_df["day"] = model_df["timestamp"].dt.day

model_df["is_weekend"] = model_df["dayofweek"].isin([5, 6]).astype(int)

print("Time features created.")

Time features created.


In [13]:
model_df = model_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

model_df["station_expanding_mean_aqi"] = (
    model_df
    .groupby("station_id")["reported_aqi"]
    .transform(lambda s: s.shift(1).expanding().mean())
)

model_df["station_expanding_std_aqi"] = (
    model_df
    .groupby("station_id")["reported_aqi"]
    .transform(lambda s: s.shift(1).expanding().std())
)

model_df["station_expanding_min_aqi"] = (
    model_df
    .groupby("station_id")["reported_aqi"]
    .transform(lambda s: s.shift(1).expanding().min())
)

model_df["station_expanding_max_aqi"] = (
    model_df
    .groupby("station_id")["reported_aqi"]
    .transform(lambda s: s.shift(1).expanding().max())
)

print("Previous-station aggregate features created.")

Previous-station aggregate features created.


In [14]:
categorical_columns = [
    "state",
    "city",
    "reported_aqi_category",
    "reported_dominant_pollutant",
    "primary_pollution_source",
    "source_attribution_confidence_category",
    "landuse_type_proxy"
]

available_categorical_columns = [
    col for col in categorical_columns
    if col in model_df.columns
]

model_encoded = pd.get_dummies(
    model_df,
    columns=available_categorical_columns,
    drop_first=False
)

print("Categorical encoding complete.")
print("Encoded shape:", model_encoded.shape)

Categorical encoding complete.
Encoded shape: (5156, 366)


In [15]:
exclude_columns = [
    "station_id",
    "station_name",
    "timestamp",
    "snapshot_fetch_time",
    "target_aqi_next"
]

feature_columns = [
    col for col in model_encoded.columns
    if col not in exclude_columns
    and pd.api.types.is_numeric_dtype(model_encoded[col])
]

print("Number of feature columns:", len(feature_columns))
print("First 40 features:")
print(feature_columns[:40])

Number of feature columns: 358
First 40 features:
['lat', 'lon', 'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3', 'aqi', 'valid_pollutant_count', 'has_particulate', 'aqi_is_valid', 'reported_aqi', 'temperature_2m', 'relative_humidity_2m', 'rain', 'wind_speed_10m', 'cloud_cover', 'weather_trapping_score', 'urban_pressure_score', 'road_density_proxy', 'environmental_risk_score', 'source_attribution_confidence', 'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_3', 'aqi_lag_6', 'aqi_lag_12', 'aqi_lag_24', 'aqi_rolling_mean_3', 'aqi_rolling_std_3', 'aqi_rolling_mean_6', 'aqi_rolling_std_6', 'aqi_rolling_mean_12', 'aqi_rolling_std_12', 'aqi_rolling_mean_24', 'aqi_rolling_std_24', 'hour', 'dayofweek']


In [16]:
model_encoded = model_encoded.sort_values("timestamp").reset_index(drop=True)

split_index = int(len(model_encoded) * 0.8)

train_df = model_encoded.iloc[:split_index].copy()
test_df = model_encoded.iloc[split_index:].copy()

X_train = train_df[feature_columns].replace([np.inf, -np.inf], np.nan)
X_test = test_df[feature_columns].replace([np.inf, -np.inf], np.nan)

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians).fillna(0)
X_test = X_test.fillna(train_medians).fillna(0)

y_train = train_df["target_aqi_next"]
y_test = test_df["target_aqi_next"]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train time range:", train_df["timestamp"].min(), "to", train_df["timestamp"].max())
print("Test time range:", test_df["timestamp"].min(), "to", test_df["timestamp"].max())

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("Train or test set is empty. Need more history.")

Train rows: 4124
Test rows: 1032
Train time range: 2026-07-15 18:00:00 to 2026-07-16 02:00:00
Test time range: 2026-07-16 02:00:00 to 2026-07-16 04:00:00


In [17]:
def evaluate_regression_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error_manual(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "model": name,
        "mae": round(float(mae), 4),
        "rmse": round(float(rmse), 4),
        "r2": round(float(r2), 4)
    }


def mean_squared_error_manual(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def get_aqi_category(aqi):
    if pd.isna(aqi):
        return "Unknown"

    aqi = float(aqi)

    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Satisfactory"
    elif aqi <= 200:
        return "Moderate"
    elif aqi <= 300:
        return "Poor"
    elif aqi <= 400:
        return "Very Poor"
    else:
        return "Severe"


print("Evaluation functions ready.")

Evaluation functions ready.


In [18]:
models = {
    "Linear Regression": LinearRegression(),

    "Ridge Regression": Ridge(
        alpha=1.0,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=18,
        min_samples_split=4,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    )
}

try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    print("XGBoost added.")

except Exception as exc:
    print("XGBoost not available. Skipping XGBoost.")
    print(exc)

print("Models prepared:")
for model_name in models:
    print("-", model_name)

XGBoost added.
Models prepared:
- Linear Regression
- Ridge Regression
- Random Forest
- Extra Trees
- Gradient Boosting
- HistGradientBoosting
- XGBoost


In [19]:
results = []
trained_models = {}
model_predictions = {}

persistence_predictions = test_df["reported_aqi"].values

results.append(
    evaluate_regression_model(
        "Persistence Baseline",
        y_test,
        persistence_predictions
    )
)

model_predictions["Persistence Baseline"] = persistence_predictions

print("Persistence Baseline result:")
print(results[-1])

Persistence Baseline result:
{'model': 'Persistence Baseline', 'mae': 15.1389, 'rmse': 47.8739, 'r2': 0.8153}


In [20]:
for model_name, model in models.items():
    print("\nTraining:", model_name)

    try:
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        trained_models[model_name] = model
        model_predictions[model_name] = predictions

        metrics = evaluate_regression_model(
            model_name,
            y_test,
            predictions
        )

        results.append(metrics)

        print("Done:", metrics)

    except Exception as exc:
        print("Failed:", model_name)
        print("Error:", exc)

print("\nAll available models tested.")


Training: Linear Regression
Done: {'model': 'Linear Regression', 'mae': 19.1134, 'rmse': 41.568, 'r2': 0.8608}

Training: Ridge Regression
Done: {'model': 'Ridge Regression', 'mae': 18.5702, 'rmse': 41.5208, 'r2': 0.8611}

Training: Random Forest
Done: {'model': 'Random Forest', 'mae': 14.5002, 'rmse': 39.879, 'r2': 0.8718}

Training: Extra Trees
Done: {'model': 'Extra Trees', 'mae': 15.5204, 'rmse': 41.618, 'r2': 0.8604}

Training: Gradient Boosting
Done: {'model': 'Gradient Boosting', 'mae': 15.7028, 'rmse': 39.6044, 'r2': 0.8736}

Training: HistGradientBoosting
Done: {'model': 'HistGradientBoosting', 'mae': 15.9481, 'rmse': 39.8933, 'r2': 0.8717}

Training: XGBoost
Done: {'model': 'XGBoost', 'mae': 15.5493, 'rmse': 40.2006, 'r2': 0.8698}

All available models tested.


In [21]:
comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    "mae",
    ascending=True
).reset_index(drop=True)

persistence_mae = comparison_df.loc[
    comparison_df["model"] == "Persistence Baseline",
    "mae"
].iloc[0]

if persistence_mae == 0:
    comparison_df["improvement_over_persistence_percent"] = 0.0
else:
    comparison_df["improvement_over_persistence_percent"] = (
        ((persistence_mae - comparison_df["mae"]) / persistence_mae) * 100
    ).round(2)

MODEL_COMPARISON_PATH = OUTPUTS_DIR / "aqi_model_comparison.csv"

comparison_df.to_csv(MODEL_COMPARISON_PATH, index=False)

print("Model comparison saved:")
print(MODEL_COMPARISON_PATH)

display(comparison_df)

Model comparison saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\aqi_model_comparison.csv


,model,mae,rmse,r2,improvement_over_persistence_percent
0,Random Forest,14.5002,39.8790,0.8718,4.22
1,Persistence Baseline,15.1389,47.8739,0.8153,0.00
2,Extra Trees,15.5204,41.6180,0.8604,-2.52
3,XGBoost,15.5493,40.2006,0.8698,-2.71
4,Gradient Boosting,15.7028,39.6044,0.8736,-3.72
5,HistGradientBoosting,15.9481,39.8933,0.8717,-5.35
6,Ridge Regression,18.5702,41.5208,0.8611,-22.67
7,Linear Regression,19.1134,41.5680,0.8608,-26.25


In [22]:
best_model_name = comparison_df.iloc[0]["model"]

best_predictions = model_predictions[best_model_name]

if best_model_name == "Persistence Baseline":
    best_model = None
else:
    best_model = trained_models[best_model_name]

print("Best model selected:", best_model_name)

display(comparison_df.iloc[[0]])

Best model selected: Random Forest


,model,mae,rmse,r2,improvement_over_persistence_percent
0,Random Forest,14.5002,39.879,0.8718,4.22


In [23]:
category_eval_df = pd.DataFrame({
    "actual_aqi": y_test.values,
    "predicted_aqi": best_predictions
})

category_eval_df["actual_category"] = category_eval_df["actual_aqi"].apply(get_aqi_category)
category_eval_df["predicted_category"] = category_eval_df["predicted_aqi"].apply(get_aqi_category)

category_accuracy = accuracy_score(
    category_eval_df["actual_category"],
    category_eval_df["predicted_category"]
)

CATEGORY_CONFUSION_PATH = OUTPUTS_DIR / "category_confusion_matrix.csv"

category_confusion = pd.crosstab(
    category_eval_df["actual_category"],
    category_eval_df["predicted_category"]
)

category_confusion.to_csv(CATEGORY_CONFUSION_PATH)

print("Best model:", best_model_name)
print("AQI category accuracy:", round(category_accuracy, 4))

display(category_confusion)

Best model: Random Forest
AQI category accuracy: 0.8905


predicted_category,Good,Moderate,Poor,Satisfactory,Severe,Very Poor
actual_category,,,,,,
Good,3,0,0,2,2,0
Moderate,0,73,20,0,0,3
Poor,0,6,143,1,0,18
Satisfactory,0,4,2,15,3,4
Severe,0,0,1,0,316,8
Very Poor,0,0,17,0,22,369


In [24]:
actual_high = (category_eval_df["actual_aqi"] > 200).astype(int)
predicted_high = (category_eval_df["predicted_aqi"] > 200).astype(int)

high_aqi_precision = precision_score(actual_high, predicted_high, zero_division=0)
high_aqi_recall = recall_score(actual_high, predicted_high, zero_division=0)
high_aqi_f1 = f1_score(actual_high, predicted_high, zero_division=0)

print("High AQI > 200 metrics:")
print("Precision:", round(high_aqi_precision, 4))
print("Recall:", round(high_aqi_recall, 4))
print("F1:", round(high_aqi_f1, 4))

High AQI > 200 metrics:
Precision: 0.9634
Recall: 0.9922
F1: 0.9776


In [26]:
# Rebuild metadata from the original non-encoded model_df
# because city/state were one-hot encoded in test_df.

metadata_df = model_df.sort_values("timestamp").reset_index(drop=True)

test_metadata_df = metadata_df.iloc[split_index:].copy()

metadata_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "timestamp",
    "reported_aqi",
    "target_aqi_next"
]

available_metadata_columns = [
    col for col in metadata_columns
    if col in test_metadata_df.columns
]

predictions_df = test_metadata_df[available_metadata_columns].copy()

predictions_df["best_model"] = best_model_name
predictions_df["predicted_aqi_next"] = best_predictions

predictions_df["residual"] = (
    predictions_df["target_aqi_next"]
    - predictions_df["predicted_aqi_next"]
)

predictions_df["absolute_error"] = predictions_df["residual"].abs()

residual_std = predictions_df["residual"].std()

if pd.isna(residual_std):
    residual_std = 0

predictions_df["predicted_aqi_p10"] = (
    predictions_df["predicted_aqi_next"] - 1.28 * residual_std
).clip(lower=0)

predictions_df["predicted_aqi_p50"] = predictions_df["predicted_aqi_next"]

predictions_df["predicted_aqi_p90"] = (
    predictions_df["predicted_aqi_next"] + 1.28 * residual_std
).clip(upper=500)

predictions_df["actual_category"] = predictions_df["target_aqi_next"].apply(get_aqi_category)
predictions_df["predicted_category"] = predictions_df["predicted_aqi_next"].apply(get_aqi_category)

PREDICTIONS_PATH = PROCESSED_DIR / "test_predictions_with_uncertainty.csv"

predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print("Predictions saved:")
print(PREDICTIONS_PATH)

print("Prediction rows:", len(predictions_df))
print("Best prediction length:", len(best_predictions))

display(predictions_df.head())

Predictions saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\test_predictions_with_uncertainty.csv
Prediction rows: 1032
Best prediction length: 1032


,station_id,station_name,city,state,timestamp,reported_aqi,target_aqi_next,best_model,predicted_aqi_next,residual,absolute_error,predicted_aqi_p10,predicted_aqi_p50,predicted_aqi_p90,actual_category,predicted_category
4124,bihar__patna__drm_office_danapur_patna_bspcb,"DRM Office Danapur, Patna - BSPCB",Patna,Bihar,2026-07-16 02:00:00,431.51,431.51,Random Forest,431.116411,0.393589,0.393589,380.542681,431.116411,481.690141,Severe,Severe
4125,delhi__delhi__nsut_jaffarpur_delhi_dpcc,"NSUT Jaffarpur, Delhi - DPCC",Delhi,Delhi,2026-07-16 02:00:00,487.55,481.32,Random Forest,488.500398,-7.180398,7.180398,437.926668,488.500398,500.000000,Severe,Severe
4126,karnataka__shivamogga__vinoba_nagara_shivamogg...,"Vinoba Nagara, Shivamogga - KSPCB",Shivamogga,Karnataka,2026-07-16 02:00:00,358.99,364.85,Random Forest,357.633139,7.216861,7.216861,307.059409,357.633139,408.206869,Very Poor,Very Poor
4127,delhi__delhi__sirifort_delhi_cpcb,"Sirifort, Delhi - CPCB",Delhi,Delhi,2026-07-16 02:00:00,312.13,323.85,Random Forest,315.536541,8.313459,8.313459,264.962811,315.536541,366.110271,Very Poor,Very Poor
4128,rajasthan__churu__subash_chowk_churu_rspcb,"Subash Chowk, Churu - RSPCB",Churu,Rajasthan,2026-07-16 02:00:00,323.85,323.85,Random Forest,321.793236,2.056764,2.056764,271.219506,321.793236,372.366966,Very Poor,Very Poor


In [27]:
FEATURE_IMPORTANCE_PATH = OUTPUTS_DIR / "feature_importance.csv"

if best_model is not None and hasattr(best_model, "feature_importances_"):
    feature_importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

elif best_model is not None and hasattr(best_model, "coef_"):
    feature_importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": np.abs(best_model.coef_)
    }).sort_values("importance", ascending=False)

elif best_model is not None:
    sample_size = min(1000, len(X_test))

    X_perm = X_test.iloc[:sample_size].copy()
    y_perm = y_test.iloc[:sample_size].copy()

    permutation_result = permutation_importance(
        best_model,
        X_perm,
        y_perm,
        n_repeats=5,
        random_state=42,
        n_jobs=-1
    )

    feature_importance_df = pd.DataFrame({
        "feature": feature_columns,
        "importance": permutation_result.importances_mean
    }).sort_values("importance", ascending=False)

else:
    feature_importance_df = pd.DataFrame({
        "feature": [],
        "importance": []
    })

feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

print("Feature importance saved:")
print(FEATURE_IMPORTANCE_PATH)

display(feature_importance_df.head(25))

Feature importance saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\feature_importance.csv


,feature,importance
9,aqi,0.404625
13,reported_aqi,0.404121
6,CO,0.040223
46,station_expanding_max_aqi,0.022897
24,aqi_lag_1,0.017086
30,aqi_rolling_mean_3,0.015417
45,station_expanding_min_aqi,0.014619
22,environmental_risk_score,0.003972
7,O3,0.003382
4,NO2,0.002981


In [28]:
if not feature_importance_df.empty:
    total_importance = feature_importance_df["importance"].abs().sum()

    lag_importance = feature_importance_df[
        feature_importance_df["feature"].str.contains(
            "lag|rolling",
            regex=True
        )
    ]["importance"].abs().sum()

    if total_importance == 0:
        lag_feature_importance_share = 0
    else:
        lag_feature_importance_share = round(
            float((lag_importance / total_importance) * 100),
            2
        )

else:
    lag_feature_importance_share = 0

print("Lag/rolling feature importance share (%):", lag_feature_importance_share)

Lag/rolling feature importance share (%): 5.09


In [29]:
per_city_error = (
    predictions_df
    .groupby(["state", "city"], as_index=False)
    .agg(
        test_rows=("absolute_error", "count"),
        mean_absolute_error=("absolute_error", "mean"),
        max_absolute_error=("absolute_error", "max"),
        mean_actual_aqi=("target_aqi_next", "mean"),
        mean_predicted_aqi=("predicted_aqi_next", "mean")
    )
)

for col in [
    "mean_absolute_error",
    "max_absolute_error",
    "mean_actual_aqi",
    "mean_predicted_aqi"
]:
    per_city_error[col] = per_city_error[col].round(2)

PER_CITY_ERROR_PATH = OUTPUTS_DIR / "per_city_error.csv"

per_city_error.to_csv(PER_CITY_ERROR_PATH, index=False)

print("Per-city error saved:")
print(PER_CITY_ERROR_PATH)

display(
    per_city_error.sort_values(
        "mean_absolute_error",
        ascending=False
    ).head(20)
)

Per-city error saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\per_city_error.csv


,state,city,test_rows,mean_absolute_error,max_absolute_error,mean_actual_aqi,mean_predicted_aqi
56,Gujarat,Vadodara,2,228.52,445.00,277.50,493.98
118,Madhya Pradesh,Sagar,2,169.37,338.22,238.10,407.47
170,Odisha,Tensa,2,163.72,309.56,218.72,364.55
120,Madhya Pradesh,Singrauli,2,126.70,239.39,219.71,346.41
231,Uttar Pradesh,Hapur,2,124.67,199.45,239.76,314.53
101,Karnataka,Mysuru,2,116.58,215.84,174.82,291.41
55,Gujarat,Surat,4,115.81,360.44,331.42,397.13
116,Madhya Pradesh,Pithampur,2,107.28,206.33,258.53,357.58
251,West Bengal,Durgapur,6,80.76,460.07,306.18,381.22
109,Madhya Pradesh,Dewas,3,78.61,192.54,246.37,299.07


In [30]:
SAVED_MODEL_DIR = MODELS_DIR / "trained_forecasting_models"
SAVED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

saved_model_paths = {}

for model_name, model in trained_models.items():
    safe_name = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    model_path = SAVED_MODEL_DIR / f"{safe_name}.pkl"

    joblib.dump(model, model_path)

    saved_model_paths[model_name] = str(model_path)

print("Saved trained models:")

for model_name, model_path in saved_model_paths.items():
    print(model_name, "→", model_path)

Saved trained models:
Linear Regression → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\linear_regression.pkl
Ridge Regression → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\ridge_regression.pkl
Random Forest → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\random_forest.pkl
Extra Trees → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\extra_trees.pkl
Gradient Boosting → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\gradient_boosting.pkl
HistGradientBoosting → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\histgradientboosting.pkl
XGBoost → C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\xgboost.pkl


In [31]:
FEATURE_COLUMNS_PATH = MODELS_DIR / "feature_columns.json"
BEST_MODEL_PATH = MODELS_DIR / "best_aqi_forecast_model.pkl"

with open(FEATURE_COLUMNS_PATH, "w", encoding="utf-8") as file:
    json.dump(feature_columns, file, indent=4)

if best_model is not None:
    joblib.dump(best_model, BEST_MODEL_PATH)
    model_saved = True
else:
    model_saved = False

print("Feature columns saved:", FEATURE_COLUMNS_PATH)
print("Best model saved:", model_saved)

if model_saved:
    print("Best model path:", BEST_MODEL_PATH)

Feature columns saved: C:\Users\Lenovo\Desktop\CleanAir_AI\models\feature_columns.json
Best model saved: True
Best model path: C:\Users\Lenovo\Desktop\CleanAir_AI\models\best_aqi_forecast_model.pkl


In [32]:
best_metrics = comparison_df.iloc[0].to_dict()

final_metrics = {
    "notebook": "04_ml_forecasting_models.ipynb",
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "training_mode": "real_time_series_history",
    "history_rows": int(len(history_df)),
    "model_rows": int(len(model_df)),
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "unique_stations": int(history_df["station_id"].nunique()),
    "unique_timestamps": int(history_df["timestamp"].nunique()),
    "models_tested": comparison_df["model"].tolist(),
    "best_model": best_model_name,
    "best_metrics": best_metrics,
    "category_accuracy": round(float(category_accuracy), 4),
    "high_aqi_threshold": 200,
    "high_aqi_precision": round(float(high_aqi_precision), 4),
    "high_aqi_recall": round(float(high_aqi_recall), 4),
    "high_aqi_f1": round(float(high_aqi_f1), 4),
    "lag_feature_importance_share_percent": lag_feature_importance_share,
    "model_comparison_file": str(MODEL_COMPARISON_PATH),
    "predictions_file": str(PREDICTIONS_PATH),
    "feature_importance_file": str(FEATURE_IMPORTANCE_PATH),
    "per_city_error_file": str(PER_CITY_ERROR_PATH),
    "feature_columns_file": str(FEATURE_COLUMNS_PATH),
    "best_model_file": str(BEST_MODEL_PATH) if model_saved else None,
    "all_saved_models": saved_model_paths
}

FINAL_METRICS_PATH = OUTPUTS_DIR / "final_aqi_model_metrics.json"

with open(FINAL_METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=4)

print("Final metrics saved:")
print(FINAL_METRICS_PATH)

print(json.dumps(final_metrics, indent=4))

Final metrics saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\final_aqi_model_metrics.json
{
    "notebook": "04_ml_forecasting_models.ipynb",
    "run_time": "2026-07-16 10:58:43",
    "training_mode": "real_time_series_history",
    "history_rows": 5644,
    "model_rows": 5156,
    "train_rows": 4124,
    "test_rows": 1032,
    "unique_stations": 488,
    "unique_timestamps": 12,
    "models_tested": [
        "Random Forest",
        "Persistence Baseline",
        "Extra Trees",
        "XGBoost",
        "Gradient Boosting",
        "HistGradientBoosting",
        "Ridge Regression",
        "Linear Regression"
    ],
    "best_model": "Random Forest",
    "best_metrics": {
        "model": "Random Forest",
        "mae": 14.5002,
        "rmse": 39.879,
        "r2": 0.8718,
        "improvement_over_persistence_percent": 4.22
    },
    "category_accuracy": 0.8905,
    "high_aqi_threshold": 200,
    "high_aqi_precision": 0.9634,
    "high_aqi_recall": 0.9922,
    "high_aqi_f1

In [33]:
required_output_files = [
    MODEL_COMPARISON_PATH,
    CATEGORY_CONFUSION_PATH,
    PREDICTIONS_PATH,
    FEATURE_IMPORTANCE_PATH,
    PER_CITY_ERROR_PATH,
    FEATURE_COLUMNS_PATH,
    FINAL_METRICS_PATH
]

if model_saved:
    required_output_files.append(BEST_MODEL_PATH)

for model_path in saved_model_paths.values():
    required_output_files.append(Path(model_path))

missing_files = [
    path for path in required_output_files
    if not Path(path).exists()
]

if missing_files:
    print("Missing files:")

    for path in missing_files:
        print(path)

    raise FileNotFoundError("Some expected output files were not created.")

print("Notebook 04 completed successfully.")
print("Created files:")

for path in required_output_files:
    print("-", path)

Notebook 04 completed successfully.
Created files:
- C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\aqi_model_comparison.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\category_confusion_matrix.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\test_predictions_with_uncertainty.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\feature_importance.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\per_city_error.csv
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\feature_columns.json
- C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\final_aqi_model_metrics.json
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\best_aqi_forecast_model.pkl
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\linear_regression.pkl
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\ridge_regression.pkl
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\random_forest.pkl
- C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\extra_trees.pk